In [1]:
import sys
import socket

print(f"Node:{socket.gethostname()}")


Node:c013.curta.zedat.fu-berlin.de


In [2]:
from pyproj import Transformer

import duckdb
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from pathlib import Path
import duckdb


from photometry.models.hapke import HapkeModel
from photometry.core.types import GeometryBatch
from photometry.fitting.least_sq import LeastSquaresFitter

import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS



In [3]:


project_root = Path.cwd().resolve()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent


# Load the RC Parquet file

parquet_path_lamo_dsk256_110825 = project_root / "data" / "silver/dsk256" / "lamo_dsk256_110825.parquet"




In [6]:
vesta_user_crs = ("+proj=eqc +lat_ts=0 +lat_0=0 +lon_0=180 "
                   "+x_0=0 +y_0=0 +R=255000 +units=m +no_defs +type=crs")
vesta_iau_crs  = "+proj=longlat +R=255000 +no_defs"

transformer = Transformer.from_crs(vesta_user_crs, vesta_iau_crs, always_xy=True)

regions_m = {
    "oppia_ejecta_orange": [
        (558714.502, -55962.360), (559476.111, -70242.536),
        (585370.830, -70813.743), (584799.623, -56343.165),
    ],
    "oppia_orange_patch": [
        (607769.687, 7328.694), (609829.494, 2542.672),
        (616735.906, 2603.254), (615827.168, 8419.181),
    ],
    "oppia_surrounding": [
        (630548.731, 11508.891), (631396.886, 1331.021),
        (648238.839, 1815.681), (647633.014, 12599.377),
    ],
}

roi_bounds = {}
for name, corners in regions_m.items():
    xs, ys = zip(*corners)
    lons, lats = transformer.transform(np.array(xs), np.array(ys))
    roi_bounds[name] = {
        "lon_min": lons.min(), "lon_max": lons.max(),
        "lat_min": lats.min(), "lat_max": lats.max(),
    }
    print(name, roi_bounds[name])

oppia_ejecta_orange {'lon_min': np.float64(-54.46281208880943), 'lon_max': np.float64(-48.4734195722745), 'lat_min': np.float64(-15.911092570290498), 'lat_max': np.float64(-12.574145253300932)}
oppia_orange_patch {'lon_min': np.float64(-43.44063536828608), 'lon_max': np.float64(-41.426021615776186), 'lat_min': np.float64(0.5713112717101492), 'lat_max': np.float64(1.8917001500264)}
oppia_surrounding {'lon_min': np.float64(-38.32242720145156), 'lon_max': np.float64(-34.34764866211577), 'lat_min': np.float64(0.2990662186011072), 'lat_max': np.float64(2.830945594487061)}


In [ ]:
for name, b in roi_bounds.items():
    roi = duckdb.sql(f"""
        SELECT pixel_x, pixel_y, image_id, iof,
               incidence, emission, phase, latitude, longitude
        FROM read_parquet('{parquet_path_lamo_dsk256_110825}')
        WHERE longitude BETWEEN {b['lon_min']} AND {b['lon_max']}
          AND latitude  BETWEEN {b['lat_min']} AND {b['lat_max']}
          AND iof > 0.01 AND image_id LIKE '%F1B%'
    """).df()
    print(name, len(roi), "pixels")

NameError: name 'parquet_path_survey_dsk256_110825' is not defined

In [8]:
for name, b in roi_bounds.items():
    roi_lamo = duckdb.sql(f"""
        SELECT COUNT(*) as n
        FROM read_parquet('{parquet_path_lamo_dsk256_110825}')
        WHERE longitude BETWEEN {b['lon_min']} AND {b['lon_max']}
          AND latitude  BETWEEN {b['lat_min']} AND {b['lat_max']}
          AND iof > 0.01 AND image_id LIKE '%F1B%'
    """).fetchone()
    print(name, roi_lamo[0], "LAMO pixels")

oppia_ejecta_orange 2854941 LAMO pixels
oppia_orange_patch 566250 LAMO pixels
oppia_surrounding 1706808 LAMO pixels


In [9]:
for name, b in roi_bounds.items():
    stats = duckdb.sql(f"""
        SELECT
            COUNT(*) AS n,
            COUNT(DISTINCT image_id) AS n_images,
            MIN(phase) AS phase_min,
            MAX(phase) AS phase_max,
            AVG(incidence) AS mean_inc,
            AVG(emission) AS mean_emi
        FROM read_parquet('{parquet_path_lamo_dsk256_110825}')
        WHERE longitude BETWEEN {b['lon_min']} AND {b['lon_max']}
          AND latitude  BETWEEN {b['lat_min']} AND {b['lat_max']}
          AND iof > 0.0156 * COS(RADIANS(incidence))
          AND image_id LIKE '%F1B%'
    """).fetchone()
    print(name, stats)

oppia_ejecta_orange (2854941, 11, 42.59602355957031, 49.269466400146484, 44.81040122823813, 15.228760581766906)
oppia_orange_patch (566252, 8, 45.95735168457031, 53.45429992675781, 53.947317590963635, 10.494354218435287)
oppia_surrounding (1706808, 10, 44.947410583496094, 52.82802200317383, 45.16595326801085, 7.058143076716103)


In [13]:
model = HapkeModel(
    enable_shoe=True, enable_roughness=True,
    fixed_parameters={'B0': 1.03, 'h': 0.04}
)
model.parameters.update({'w': 0.4656, 'g': -0.3325, 'theta_bar': 8.38})

# their exact CRS, for writing output back in their coordinate system
vesta_user_crs = CRS.from_proj4(
    "+proj=eqc +lat_ts=0 +lat_0=0 +lon_0=180 "
    "+x_0=0 +y_0=0 +R=255000 +units=m +no_defs"
)

In [14]:

def correct_and_export_region(name, lon_min, lon_max, lat_min, lat_max,
                                x_min, x_max, y_min, y_max,
                                parquet_path, out_path, grid_res_m=20):
    """
    lon/lat bounds for the DuckDB query,
    x/y (meters, user CRS) bounds for the GeoTIFF transform.
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    df = duckdb.sql(f"""
        SELECT pixel_x, pixel_y, iof, incidence, emission, phase,
               latitude, longitude
        FROM read_parquet('{parquet_path}')
        WHERE longitude BETWEEN {lon_min} AND {lon_max}
          AND latitude  BETWEEN {lat_min} AND {lat_max}
          AND iof > 0.0156 * COS(RADIANS(incidence))
          AND image_id LIKE '%F1B%'
    """).df()
    print(f"{name}: {len(df):,} pixels loaded")

    geom = GeometryBatch(
        incidence=np.deg2rad(df['incidence'].to_numpy()),
        emission=np.deg2rad(df['emission'].to_numpy()),
        phase=np.deg2rad(df['phase'].to_numpy())
    )
    geom_std = GeometryBatch(
        incidence=np.deg2rad(np.full(len(df), 30.0)),
        emission=np.deg2rad(np.zeros(len(df))),
        phase=np.deg2rad(np.full(len(df), 30.0))
    )
    pred_real = model._reflectance_numpy(geom)
    pred_std  = model._reflectance_numpy(geom_std)
    correction = np.clip(pred_std / np.maximum(pred_real, 1e-6), 0.5, 2.5)
    df['iof_corrected'] = df['iof'].to_numpy() * correction

    # grid onto a regular lat/lon mesh at fixed physical resolution
    n_lat = int((lat_max - lat_min) * 111000 / grid_res_m)
    n_lon = int((lon_max - lon_min) * 111000 *
                np.cos(np.deg2rad((lat_min+lat_max)/2)) / grid_res_m)
    n_lat, n_lon = max(n_lat, 10), max(n_lon, 10)

    from scipy.stats import binned_statistic_2d
    grid, _, _, _ = binned_statistic_2d(
        df['latitude'], df['longitude'], df['iof_corrected'],
        statistic='mean', bins=[n_lat, n_lon],
        range=[[lat_min, lat_max], [lon_min, lon_max]]
    )
    grid = np.flipud(grid)  # north-up for raster convention

    transform = from_bounds(x_min, y_min, x_max, y_max, n_lon, n_lat)

    with rasterio.open(
        out_path, "w", driver="GTiff",
        height=n_lat, width=n_lon, count=1,
        dtype="float32", crs=vesta_user_crs, transform=transform,
        nodata=np.nan
    ) as dst:
        dst.write(grid.astype("float32"), 1)
    print(f"  → wrote {out_path}")
    return df, grid


# Region 1: Oppia ejecta orange

In [21]:
Path("results_tif").mkdir(parents=True, exist_ok=True)

In [22]:
# Region 1: Oppia ejecta orange
r1_df, r1_grid = correct_and_export_region(
    "oppia_ejecta_orange",
    lon_min=roi_bounds['oppia_ejecta_orange']['lon_min'],
    lon_max=roi_bounds['oppia_ejecta_orange']['lon_max'],
    lat_min=roi_bounds['oppia_ejecta_orange']['lat_min'],
    lat_max=roi_bounds['oppia_ejecta_orange']['lat_max'],
    x_min=558714.502, x_max=585370.830,
    y_min=-70813.743, y_max=-55962.360,
    parquet_path= str(parquet_path_lamo_dsk256_110825),
    out_path="results_tif/roi_oppia_ejecta_orange_corrected.tif"
)


oppia_ejecta_orange: 2,854,941 pixels loaded


: 

: 

: 

# Region 2: Oppia orange patch